# Structured Output

Force Jockey to return typed JSON matching a schema you define. Essential for programmatic consumption and agent integrations.

In [ ]:
import json
import os
import time

import requests

# Configuration
API_KEY = os.environ.get("TWELVELABS_API_KEY", "YOUR_API_KEY")
BASE_URL = "https://api.twelvelabs.io/v1.3"
HEADERS = {"x-api-key": API_KEY, "Content-Type": "application/json"}

# Replace with your knowledge store ID
STORE_ID = "your_knowledge_store_id"

## When You Need This

- Building pipelines that parse response data programmatically
- Agent-to-agent communication where typed output matters
- Extracting structured metadata from video collections

## Helper: Parse Response

A reusable helper to extract text content from Jockey API responses.

In [ ]:
def parse_response(result: dict) -> str | dict:
    """Extract text content from a Jockey API response.

    Args:
        result: The parsed JSON response from the API.

    Returns:
        The text content string, or an empty string if not found.
    """
    for output in result["output"]:
        if output["type"] == "message":
            for content in output["content"]:
                return content["text"]
    return ""

## Basic Usage

Pass a JSON Schema via the `text` parameter to constrain the response format. The schema defines the structure Jockey must follow.

In [ ]:
response = requests.post(
    f"{BASE_URL}/responses",
    headers=HEADERS,
    json={
        "model": "jockey1.0",
        "input": [
            {
                "type": "message",
                "role": "user",
                "content": "List the main themes and key entities in these videos",
            }
        ],
        "knowledge_store_id": STORE_ID,
        "text": {
            "format": {
                "type": "json_schema",
                "name": "themes_and_entities",
                "schema": {
                    "type": "object",
                    "properties": {
                        "themes": {
                            "type": "array",
                            "items": {"type": "string"},
                        },
                        "entities": {
                            "type": "array",
                            "items": {
                                "type": "object",
                                "properties": {
                                    "name": {"type": "string"},
                                    "type": {"type": "string"},
                                    "frequency": {"type": "string"},
                                },
                            },
                        },
                    },
                },
            },
        },
    },
)

result = response.json()
text = parse_response(result)

# The text is a JSON string -- parse it
structured = json.loads(text)
print(f"Themes: {structured['themes']}")
for entity in structured["entities"]:
    print(f"  {entity['name']} ({entity['type']})")

## Example: Video Catalog Extraction

Extract structured metadata for every video in a collection.

In [ ]:
CATALOG_SCHEMA = {
    "type": "object",
    "properties": {
        "videos": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "title": {"type": "string"},
                    "duration_estimate": {"type": "string"},
                    "primary_topic": {"type": "string"},
                    "mood": {"type": "string"},
                    "tags": {"type": "array", "items": {"type": "string"}},
                },
            },
        },
        "collection_summary": {"type": "string"},
    },
}

response = requests.post(
    f"{BASE_URL}/responses",
    headers=HEADERS,
    json={
        "model": "jockey1.0",
        "input": [
            {
                "type": "message",
                "role": "user",
                "content": "Catalog all videos with metadata",
            }
        ],
        "knowledge_store_id": STORE_ID,
        "text": {"format": {"type": "json_schema", "name": "video_catalog", "schema": CATALOG_SCHEMA}},
    },
)

catalog = json.loads(parse_response(response.json()))
print(f"Collection summary: {catalog.get('collection_summary', 'N/A')}")
print(f"Videos found: {len(catalog.get('videos', []))}")
for video in catalog.get("videos", []):
    print(f"  - {video['title']} | Topic: {video['primary_topic']} | Tags: {video['tags']}")

## Example: Content Enrichment

Use `instructions` to apply a domain-specific lens, with structured output to capture the enriched analysis.

| Context | Instructions |
|---------|-------------|
| Accessibility | "Analyze for accessibility: describe visual elements for screen readers, note caption quality, identify audio-only content" |
| Compliance | "Review for regulatory compliance: identify claims, disclaimers, required disclosures" |
| Education | "Analyze pedagogical effectiveness: identify learning objectives, teaching methods, assessment opportunities" |
| SEO | "Extract SEO metadata: keywords, descriptions, suggested titles, topic clusters" |

In [ ]:
ENRICHMENT_SCHEMA = {
    "type": "object",
    "properties": {
        "enrichments": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "video_reference": {"type": "string"},
                    "original_summary": {"type": "string"},
                    "enriched_analysis": {"type": "string"},
                    "new_insights": {
                        "type": "array",
                        "items": {"type": "string"},
                    },
                    "tags": {
                        "type": "array",
                        "items": {"type": "string"},
                    },
                },
            },
        }
    },
}

response = requests.post(
    f"{BASE_URL}/responses",
    headers=HEADERS,
    json={
        "model": "jockey1.0",
        "instructions": (
            "You are a brand strategist. Analyze videos through the lens of "
            "brand perception and marketing effectiveness."
        ),
        "input": [
            {
                "type": "message",
                "role": "user",
                "content": (
                    "For each video, provide enriched analysis focusing on "
                    "brand messaging effectiveness, audience engagement signals, "
                    "and production quality."
                ),
            }
        ],
        "knowledge_store_id": STORE_ID,
        "text": {"format": {"type": "json_schema", "name": "enrichment", "schema": ENRICHMENT_SCHEMA}},
    },
)

enrichments = json.loads(parse_response(response.json()))
for item in enrichments.get("enrichments", []):
    print(f"Video: {item['video_reference']}")
    print(f"  Analysis: {item['enriched_analysis'][:100]}...")
    print(f"  Insights: {item['new_insights']}")
    print()

## Example: Organization Discovery

Let Jockey recommend the best ways to categorize your collection. Each axis is scored by how well it separates content into distinct, useful groups.

In [ ]:
AXES_SCHEMA = {
    "type": "object",
    "properties": {
        "axes": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "axis": {"type": "string"},
                    "description": {"type": "string"},
                    "expected_groups": {"type": "integer"},
                    "example_categories": {
                        "type": "array",
                        "items": {"type": "string"},
                    },
                    "effectiveness_score": {"type": "number"},
                    "rationale": {"type": "string"},
                },
            },
        },
        "recommendation": {"type": "string"},
    },
}

response = requests.post(
    f"{BASE_URL}/responses",
    headers=HEADERS,
    json={
        "model": "jockey1.0",
        "instructions": (
            "You are a content strategist. Analyze the video collection and "
            "recommend the most effective ways to organize it. Score each axis "
            "by how well it separates content into distinct, useful groups."
        ),
        "input": [
            {
                "type": "message",
                "role": "user",
                "content": (
                    "What are the top 5 best ways to organize this video collection? "
                    "Score each from 0-1 based on how cleanly it separates the content."
                ),
            }
        ],
        "knowledge_store_id": STORE_ID,
        "text": {"format": {"type": "json_schema", "name": "organization_axes", "schema": AXES_SCHEMA}},
    },
)

org = json.loads(parse_response(response.json()))
print(f"Recommendation: {org.get('recommendation', 'N/A')}\n")
for axis in org.get("axes", []):
    print(f"  {axis['axis']} (score: {axis['effectiveness_score']})")
    print(f"    {axis['description']}")
    print(f"    Example categories: {axis['example_categories']}")
    print()

## Common Pitfalls

- **Response text is a JSON string** -- you still need to call `json.loads()` on the text content
- **Schema must be valid JSON Schema** -- invalid schemas will cause errors
- **Keep schemas simple** -- deeply nested schemas may produce less reliable output

## Next Steps

- [Streaming](./streaming.ipynb) -- Receive responses in real-time via SSE
- [Multi-Turn Sessions](./multi_turn_sessions.ipynb) -- Continue conversations across multiple requests
- [Error Handling](./error_handling.ipynb) -- Retry strategies, polling helpers, and common error patterns
- [API Reference: POST /responses](https://twelvelabs-preview-7f6af7ac-b5df-4358-a7dc-8573e931a808.docs.buildwithfern.com/api-reference/responses/create-response)